# WiDS Global Datathon 2026 — Survival Fusion

This notebook trains a survival‑aware ensemble to predict hit probabilities at 12h/24h/48h/72h.

**Pipeline:**
- **GBSA** (Gradient Boosting Survival Analysis) multi‑seed ensemble for survival ranking.
- **LightGBM** per‑horizon classifiers (24h, 48h) with **IPCW** to handle censoring.
- **Fixed blend** for 24h/48h: combine GBSA + LGBM.
- **72h** uses a distance‑based sigmoid heuristic.
- Final probabilities are **monotonic** across horizons.



## Problem Framing
We predict the probability that a wildfire reaches an evacuation zone by 12h/24h/48h/72h.
The metric combines ranking quality (C-index) and calibration (Brier).
Survival analysis handles censoring: if a fire never hits within 72h, it is right-censored.


## Tuning Tips (Optional)
- Enable `AUTO_TUNE=True` to search a small grid of blend weights and sigmoid params using OOF.
- Keep the grid small to avoid overfitting.
- If you improve OOF but lose leaderboard score, revert to fixed weights.


## Why This Pipeline
- **GBSA** captures survival ordering (good for C-index).
- **IPCW LightGBM** improves probability calibration at 24h/48h.
- A fixed blend keeps the model stable and reduces overfitting risk.
- **72h** uses a distance-based sigmoid to avoid noisy long-horizon fitting.


## Tuning Tips (Optional)
- Enable `AUTO_TUNE=True` to search a small grid of blend weights and sigmoid params using OOF.
- Keep the grid small to avoid overfitting.
- If you improve OOF but lose leaderboard score, revert to fixed weights.


## 72h Insight
The 72h label is tricky: many samples are censored before 72h and excluded from the Brier calculation.
This often makes **higher 72h probabilities** safer for the metric.
We provide a switch to control 72h with either:
- `sigmoid` (distance-based),
- `gbsa` (survival model),
- `blend` (mix both), or
- `constant1` (all ones).
If your 72h predictions look too low, try `blend` or `constant1`.


## Experiment Log
| Run ID | RUN_MODE | W24 | W48 | P24 | P72_MODE | P72_BLEND_W | SIG_THR | SIG_SCALE | P72_FLOOR | P72_POWER | GBSA_FEATS | LGB_FEATS | OOF Hybrid | Public LB | Notes |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 0.97080 | full | 0.95 | 0.45 | 1.1 | sigmoid | - | 5450 | 50 | 0.0 | 1.0 | raw | engineered | 0.97xx | 0.97080 | baseline |
| 0.97073 | full | auto | auto | auto | blend | auto | 5450 | 50 | auto | auto | raw | engineered | 0.97xx | 0.97073 | AUTO_TUNE, 72h=blend |
| 0.97085 | full | auto | auto | auto | constant1 | - | 5450 | 50 | 0.0 | 1.0 | raw | engineered | 0.97xx | 0.97085 | AUTO_TUNE, 72h=constant1 |
| 0.96986 | full | 0.95 | 0.45 | 1.1 | sigmoid | - | 5450 | 50 | 0.0 | 1.0 | engineered | engineered | 0.96xx | 0.96986 | GBSA on engineered features |
| NEXT | full | 0.95 | 0.45 | 1.1 | blend | 0.6 | 5450 | 50 | 0.6 | 0.8 | engineered | minimal |  |  | low-dim + lifted 72h |
|  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |


## Next Improvements (Ideas)
- **GBSA with engineered features**: run GBSA on the same feature set used for LGBM (distance transforms + kinematics).
- **72h calibration**: try `P72_MODE="blend"` with a higher floor (e.g., `P72_FLOOR=0.6`) or a power < 1.0 to lift low probabilities.
- **Blend tuning**: keep auto-tune grid small to avoid overfitting. Use the best OOF weights, then fix them for the final run.


In [1]:
#!/usr/bin/env python3
"""
WiDS Global Datathon 2026 — Survival Fusion
==========================================

Pipeline summary:
1) GBSA multi-seed survival ensemble
2) LightGBM per-horizon classifiers with IPCW weighting
3) Fixed-weight blend at 24h/48h
4) 72h probability from a distance-based sigmoid
5) Monotonicity enforcement across horizons

"""

import os
import sys
import subprocess
import warnings
from typing import List, Tuple

warnings.filterwarnings("ignore")

DATA_DIR = "/kaggle/input/competitions/WiDSWorldWide_GlobalDathon26"
OUTPUT_PATH = "/kaggle/working/submission.csv"

# Run mode
RUN_MODE = "full"  # "full" or "fast"
DO_OOF = True
CV_BAG_TEST = True

# GBSA feature set
GBSA_FEATURES = "engineered"  # "raw" or "engineered"

# Feature set for LightGBM
FEATURE_SET = "minimal"  # "engineered" or "minimal"

# Minimal feature list (low-dimensional, stable)
MINIMAL_FEATURES = [
    "dist_min_ci_0_5h",
    "dist_km",
    "log_distance",
    "closing_speed_m_per_h",
    "radial_growth_rate_m_per_h",
    "num_perimeters_0_5h",
    "area_first_ha",
    "area_growth_rate_ha_per_h",
    "alignment_abs",
    "eta_effective",
]
  # use fold models to predict test (as in reference)

# Blend weights (from reference)
W_GBSA_24 = 0.95
W_GBSA_48 = 0.45  # try 0.55 if you want alternative
POWER_CAL_24 = 1.1  # power calibration for GBSA 24h

# Sigmoid override for 72h (distance to evac)
SIGMOID_THR = 5450  # meters
SIGMOID_SCALE = 50


# 72h control
P72_MODE = "blend"  # "sigmoid", "gbsa", "blend", "constant1"
P72_BLEND_W = 0.6     # only used if P72_MODE="blend" (weight on GBSA 72h)
P72_FLOOR = 0.6       # optional floor for 72h predictions
P72_CONST = 0.9       # used when P72_MODE="constant"
P72_POWER = 0.8       # <1.0 lifts low 72h probs; >1.0 compresses



# Optional: auto-tune blend weights on OOF (small grid)
AUTO_TUNE = False
TUNE_GRID = {
    "W_GBSA_24": [0.90, 0.95, 0.97],
    "W_GBSA_48": [0.40, 0.45, 0.50, 0.55],
    "POWER_CAL_24": [1.0, 1.05, 1.1],
    "SIGMOID_THR": [5200, 5450, 5700],
    "SIGMOID_SCALE": [40, 50, 60],
}

# Seeds
GBSA_SEEDS_FULL = (
    123, 456, 789, 777, 666,
    1511, 1523, 2025, 2026, 2033,
    279, 239, 70, 77, 31,
    2024, 2077, 3077, 123456, 654321,
    4640, 841, 7755, 8525, 2701,
    8817, 8864, 4085, 8919, 934,
    4746, 1699, 7401, 7826, 4098,
    2921, 1204, 2752, 8384, 1284,
)
GBSA_SEEDS_FAST = tuple(range(42, 52))

LGB_SEEDS_FULL = (
    123, 456, 789, 777, 666,
    1511, 1523, 2025, 2026, 2033,
    279, 239, 70, 77, 31,
    2024, 2077, 3077, 123456, 654321,
    2034, 2035, 2036, 1984, 1991,
    3255, 1011, 6241, 2790, 6847,
    8141, 7752, 432, 906, 6217,
    7785, 1603, 7609, 965, 2506,
    3771, 7080, 4963, 7939, 2751,
    473, 339, 3675, 5535, 4760,
)
LGB_SEEDS_FAST = tuple(range(42, 52))

# Horizons
HORIZONS_PRED = [12, 24, 48, 72]

In [2]:
# ─────────────────────────────────────────────────────────────────────────────
# Optional installs (Kaggle-safe)
# ─────────────────────────────────────────────────────────────────────────────
def _install(pkg: str, import_name: str = None):
    name = import_name or pkg
    try:
        __import__(name)
    except Exception:
        print(f"[INSTALL] {pkg}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])


_install("scikit-survival", "sksurv")
_install("lightgbm")
_install("scikit-learn", "sklearn")

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
import lightgbm as lgb
from sksurv.util import Surv
from sksurv.ensemble import GradientBoostingSurvivalAnalysis


# ─────────────────────────────────────────────────────────────────────────────
# Load data
# ─────────────────────────────────────────────────────────────────────────────
train_df = pd.read_csv(f"{DATA_DIR}/train.csv")
test_df = pd.read_csv(f"{DATA_DIR}/test.csv")
sample_sub = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")

print("Train:", train_df.shape, "Test:", test_df.shape)
print("Events:", train_df["event"].value_counts().to_dict())

[INSTALL] scikit-survival
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 78.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.1/222.1 kB 10.2 MB/s eta 0:00:00
Train: (221, 37) Test: (95, 35)
Events: {0: 152, 1: 69}


## Feature Engineering

In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# Feature engineering (for LightGBM)
# Notes:
# - Focus on distance-related transforms (dominant signal in this dataset).
# - Kinematic + zone + temporal signals add diversity.
# ─────────────────────────────────────────────────────────────────────────────
def create_features(df: pd.DataFrame) -> pd.DataFrame:
    """Feature engineering for LightGBM classifiers."""
    result = df.copy()
    dist = result["dist_min_ci_0_5h"].clip(lower=1)
    speed = result["closing_speed_m_per_h"]
    perimeters = result["num_perimeters_0_5h"]
    area_first = result["area_first_ha"]

    # Distance transformations
    result["log_distance"] = np.log1p(dist)
    result["inv_distance"] = 1 / (dist / 1000 + 0.1)
    result["inv_distance_sq"] = result["inv_distance"] ** 2
    result["sqrt_distance"] = np.sqrt(dist)
    result["dist_km"] = dist / 1000
    result["dist_km_sq"] = (dist / 1000) ** 2
    result["dist_rank"] = dist.rank(pct=True)

    # Area-to-distance
    fire_radius = np.sqrt(area_first * 10000 / np.pi)
    result["radius_to_dist"] = fire_radius / dist
    result["area_to_dist_ratio"] = area_first / (dist / 1000 + 0.1)
    result["log_area_dist_ratio"] = np.log1p(area_first) - np.log1p(dist)

    # Movement / kinematics
    result["has_movement"] = (perimeters > 1).astype(float)
    closing_pos = speed.clip(lower=0)
    result["eta_hours"] = np.where(closing_pos > 0.01, dist / closing_pos, 9999).clip(max=9999)
    result["log_eta"] = np.log1p(result["eta_hours"].clip(0, 9999))
    radial_growth = result["radial_growth_rate_m_per_h"].clip(lower=0)
    effective_closing = closing_pos + radial_growth
    result["effective_closing_speed"] = effective_closing
    result["eta_effective"] = np.where(effective_closing > 0.01, dist / effective_closing, 9999).clip(max=9999)
    result["threat_score"] = result["alignment_abs"] * speed / np.log1p(dist)
    result["fire_urgency"] = perimeters * speed
    result["growth_intensity"] = result["area_growth_rate_ha_per_h"] * perimeters

    # Zones
    result["zone_critical"] = (dist < 5000).astype(float)
    result["zone_warning"] = ((dist >= 5000) & (dist < 10000)).astype(float)
    result["zone_safe"] = (dist >= 10000).astype(float)

    # Temporal
    result["is_summer"] = result["event_start_month"].isin([6, 7, 8]).astype(float)
    result["is_afternoon"] = ((result["event_start_hour"] >= 12) & (result["event_start_hour"] < 20)).astype(float)

    # Drop highly redundant
    drop_cols = [
        "relative_growth_0_5h",
        "projected_advance_m",
        "centroid_displacement_m",
        "centroid_speed_m_per_h",
        "closing_speed_abs_m_per_h",
        "area_growth_abs_0_5h",
    ]
    result = result.drop(columns=[c for c in drop_cols if c in result.columns])
    result = result.replace([np.inf, -np.inf], np.nan).fillna(0)
    return result


train_processed = create_features(train_df)
test_processed = create_features(test_df)

print("Engineered features (LGB):", len([c for c in train_processed.columns if c not in ["event_id", "event", "time_to_hit_hours"]]))

def select_feature_set(df, feature_set):
    """Return a DataFrame with the chosen feature subset."""
    if feature_set == "minimal":
        cols = [c for c in MINIMAL_FEATURES if c in df.columns]
        return df[cols].copy()
    return df.copy()


Engineered features (LGB): 51


## Metric Competition

In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# Metrics + helpers (competition metric)
# ─────────────────────────────────────────────────────────────────────────────
def compute_c_index(time, event, risk):
    """Harrell's C-index for survival ranking."""
    n = len(time)
    concordant = comparable = 0
    for i in range(n):
        if event[i] != 1:
            continue
        for j in range(n):
            if i == j or time[i] >= time[j]:
                continue
            comparable += 1
            if risk[i] > risk[j]:
                concordant += 1
            elif risk[i] == risk[j]:
                concordant += 0.5
    return concordant / comparable if comparable > 0 else 0.5


def compute_brier(time, event, prob, horizon):
    """Censor-aware Brier score at a fixed horizon."""
    valid = ~((event == 0) & (time < horizon))
    if valid.sum() == 0:
        return 0.25
    y_true = ((event == 1) & (time <= horizon)).astype(float)[valid]
    return float(np.mean((np.clip(prob[valid], 0, 1) - y_true) ** 2))


def compute_hybrid_score(time, event, p24, p48, p72):
    """Hybrid = 0.3*C-index + 0.7*(1 - Weighted Brier)."""
    risk = 0.3 * p24 + 0.4 * p48 + 0.3 * p72
    c_index = compute_c_index(time, event, risk)
    b24 = compute_brier(time, event, p24, 24)
    b48 = compute_brier(time, event, p48, 48)
    b72 = compute_brier(time, event, p72, 72)
    wb = 0.3 * b24 + 0.4 * b48 + 0.3 * b72
    return 0.3 * c_index + 0.7 * (1 - wb), c_index, wb


def enforce_monotonicity(preds):
    """Ensure probabilities are non-decreasing across horizons."""
    result = np.clip(preds, 0, 1)
    for i in range(1, result.shape[1]):
        result[:, i] = np.maximum(result[:, i], result[:, i - 1])
    return result


def get_surv_predictions(model, X):
    """Convert survival function predictions into P(hit by horizon)."""
    surv_fns = model.predict_survival_function(X)
    preds = np.empty((len(surv_fns), len(HORIZONS_PRED)), dtype=float)
    for i, fn in enumerate(surv_fns):
        t_min, t_max = fn.domain
        preds[i, :] = fn(np.clip(HORIZONS_PRED, t_min, t_max))
    return 1.0 - preds


def sigmoid_pred(dist, threshold, scale):
    """Distance-based calibration for 72h probability."""
    return 1.0 / (1.0 + np.exp((dist - threshold) / scale))


def make_binary_target(time_vals, event_vals, horizon):
    """Binary target for per-horizon classification with censoring mask."""
    unknown = (event_vals == 0) & (time_vals < horizon)
    y = ((event_vals == 1) & (time_vals <= horizon)).astype(float)
    return y, ~unknown


def compute_ipcw_weights(times, events, horizon):
    """Compute IPCW weights from Kaplan-Meier censoring distribution."""
    unique_t = np.sort(np.unique(times))
    surv = np.ones(len(unique_t))
    for i, t in enumerate(unique_t):
        at_risk = (times >= t).sum()
        censored_at_t = ((times == t) & (events == 0)).sum()
        if at_risk > 0:
            surv[i] = 1 - censored_at_t / at_risk
        if i > 0:
            surv[i] *= surv[i - 1]

    def G(t):
        idx = np.searchsorted(unique_t, t, side="right") - 1
        return max(surv[idx], 0.01) if idx >= 0 else 1.0

    weights = np.ones(len(times))
    for i in range(len(times)):
        if events[i] == 1 and times[i] <= horizon:
            weights[i] = 1.0 / G(times[i])
        elif times[i] >= horizon:
            weights[i] = 1.0 / G(horizon)
    return weights

def make_p72(dist, gbsa_p72=None):
    """Compute 72h probability based on selected mode."""
    sig = sigmoid_pred(dist, SIGMOID_THR, SIGMOID_SCALE)
    if P72_MODE == "constant1":
        p = np.ones_like(sig)
    elif P72_MODE == "constant":
        p = np.full_like(sig, P72_CONST)
    elif P72_MODE == "gbsa" and gbsa_p72 is not None:
        p = gbsa_p72
    elif P72_MODE == "blend" and gbsa_p72 is not None:
        p = P72_BLEND_W * gbsa_p72 + (1 - P72_BLEND_W) * sig
    else:
        p = sig
    if P72_POWER and P72_POWER != 1.0:
        p = np.clip(p, 0, 1) ** P72_POWER
    if P72_FLOOR > 0:
        p = np.clip(p, P72_FLOOR, 1.0)
    return np.clip(p, 0.0, 1.0)



## Training and Config

In [5]:
# ─────────────────────────────────────────────────────────────────────────────
# GBSA Survival Ensemble
# - Multi-seed + multi-config survival boosting
# - Produces P(hit by 12/24/48/72) directly from survival functions
# ─────────────────────────────────────────────────────────────────────────────
if GBSA_FEATURES == "engineered":
    X_surv_train = train_processed.drop(columns=["event_id", "event", "time_to_hit_hours"])
    X_surv_test = test_processed.drop(columns=["event_id"])
else:
    X_surv_train = train_df.drop(columns=["event_id", "event", "time_to_hit_hours"])
    X_surv_test = test_df.drop(columns=["event_id"])
y_surv = Surv.from_arrays(event=train_df["event"].astype(bool), time=train_df["time_to_hit_hours"])
event_values = train_df["event"].values
time_values = train_df["time_to_hit_hours"].values
dist_train = train_df["dist_min_ci_0_5h"].values
dist_test = test_df["dist_min_ci_0_5h"].values

gbsa_configs = [
    {"learning_rate": 0.01, "subsample": 0.7,  "max_depth": 3, "min_samples_leaf": 12, "min_samples_split": 3, "n_estimators": 1200, "dropout_rate": 0.0},
    {"learning_rate": 0.01, "subsample": 0.85, "max_depth": 3, "min_samples_leaf": 15, "min_samples_split": 3, "n_estimators": 1200, "dropout_rate": 0.0},
    {"learning_rate": 0.01, "subsample": 0.6,  "max_depth": 3, "min_samples_leaf": 12, "min_samples_split": 3, "n_estimators": 1200, "dropout_rate": 0.0},
    {"learning_rate": 0.005,"subsample": 0.85, "max_depth": 3, "min_samples_leaf": 12, "min_samples_split": 3, "n_estimators": 2000, "dropout_rate": 0.0},
    {"learning_rate": 0.01, "subsample": 0.85, "max_depth": 3, "min_samples_leaf": 20, "min_samples_split": 3, "n_estimators": 1400, "dropout_rate": 0.0},
]

GBSA_SEEDS = GBSA_SEEDS_FULL if RUN_MODE == "full" else GBSA_SEEDS_FAST

oof_gbsa = np.zeros((len(X_surv_train), 4)) if DO_OOF else None
test_gbsa = np.zeros((len(X_surv_test), 4))

print(f"GBSA configs: {len(gbsa_configs)} | seeds: {len(GBSA_SEEDS)} | CV-bag test: {CV_BAG_TEST}")

for cfg_idx, cfg in enumerate(gbsa_configs, 1):
    cfg_oof = np.zeros((len(X_surv_train), 4)) if DO_OOF else None
    cfg_test = np.zeros((len(X_surv_test), 4))
    for seed in GBSA_SEEDS:
        seed_oof = np.zeros((len(X_surv_train), 4)) if DO_OOF else None
        seed_test = np.zeros((len(X_surv_test), 4))
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
        for tr_idx, va_idx in cv.split(X_surv_train, event_values):
            m = GradientBoostingSurvivalAnalysis(**{**cfg, "random_state": seed})
            m.fit(X_surv_train.iloc[tr_idx], y_surv[tr_idx])
            if DO_OOF:
                seed_oof[va_idx] = get_surv_predictions(m, X_surv_train.iloc[va_idx])
            if CV_BAG_TEST:
                seed_test += get_surv_predictions(m, X_surv_test) / 5
        if DO_OOF:
            cfg_oof += seed_oof / len(GBSA_SEEDS)
        if CV_BAG_TEST:
            cfg_test += seed_test / len(GBSA_SEEDS)
        else:
            m_full = GradientBoostingSurvivalAnalysis(**{**cfg, "random_state": seed})
            m_full.fit(X_surv_train, y_surv)
            cfg_test += get_surv_predictions(m_full, X_surv_test) / len(GBSA_SEEDS)
    if DO_OOF:
        oof_gbsa += cfg_oof / len(gbsa_configs)
    test_gbsa += cfg_test / len(gbsa_configs)
    print(f"  Config {cfg_idx}/{len(gbsa_configs)} done (n={cfg['n_estimators']}, ss={cfg['subsample']})")

# Store raw GBSA predictions (apply power-cal later)
oof_gbsa_raw = oof_gbsa.copy() if DO_OOF else None
test_gbsa_raw = test_gbsa.copy()


# ─────────────────────────────────────────────────────────────────────────────
# LightGBM IPCW per-horizon
# - Separate classifiers for 24h and 48h
# - IPCW corrects censoring bias
# ─────────────────────────────────────────────────────────────────────────────
X_lgb_train = train_processed.drop(columns=["event_id", "event", "time_to_hit_hours"])
X_lgb_test = test_processed.drop(columns=["event_id"])

# Apply low-dimensional feature set if requested
if FEATURE_SET == "minimal":
    X_lgb_train = select_feature_set(X_lgb_train, "minimal")
    X_lgb_test = select_feature_set(X_lgb_test, "minimal")

lgb_cfgs = {
    24: {
        "max_depth": 3, "learning_rate": 0.03, "n_estimators": 300,
        "subsample": 0.7, "colsample_bytree": 0.7, "min_child_samples": 8,
        "reg_alpha": 0.5, "reg_lambda": 2.0, "num_leaves": 7,
    },
    48: {
        "max_depth": 2, "learning_rate": 0.05, "n_estimators": 200,
        "subsample": 0.8, "colsample_bytree": 0.8, "min_child_samples": 5,
        "reg_alpha": 0.1, "reg_lambda": 1.0, "num_leaves": 4,
    },
}

LGB_SEEDS = LGB_SEEDS_FULL if RUN_MODE == "full" else LGB_SEEDS_FAST
N_LGB_SEEDS = len(LGB_SEEDS)
lgb_oof, lgb_test = {}, {}

print(f"LGB seeds: {N_LGB_SEEDS}")

for horizon in [24, 48]:
    y_bin, mask = make_binary_target(time_values, event_values, horizon)
    valid_idx = np.where(mask)[0]
    cfg = lgb_cfgs[horizon]
    all_oof = np.zeros(len(X_lgb_train))
    all_test = np.zeros(len(X_lgb_test))
    for seed in LGB_SEEDS:
        seed_oof = np.zeros(len(X_lgb_train))
        last_model = None
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
        for tr_v, va_v in cv.split(valid_idx, y_bin[mask]):
            tr_idx, va_idx = valid_idx[tr_v], valid_idx[va_v]
            ipcw_w = compute_ipcw_weights(time_values[tr_idx], event_values[tr_idx], horizon)
            m = lgb.LGBMClassifier(**cfg, objective="binary", random_state=seed, verbose=-1)
            m.fit(X_lgb_train.iloc[tr_idx], y_bin[tr_idx], sample_weight=ipcw_w)
            seed_oof[va_idx] = m.predict_proba(X_lgb_train.iloc[va_idx])[:, 1]
            last_model = m
        censored_idx = np.where(~mask)[0]
        if len(censored_idx) > 0 and last_model is not None:
            seed_oof[censored_idx] = last_model.predict_proba(X_lgb_train.iloc[censored_idx])[:, 1]
        all_oof += seed_oof

        # Full-data model for test
        ipcw_w_full = compute_ipcw_weights(time_values[valid_idx], event_values[valid_idx], horizon)
        m_full = lgb.LGBMClassifier(**cfg, objective="binary", random_state=seed, verbose=-1)
        m_full.fit(X_lgb_train.iloc[valid_idx], y_bin[valid_idx], sample_weight=ipcw_w_full)
        all_test += m_full.predict_proba(X_lgb_test)[:, 1]

    lgb_oof[horizon] = all_oof / N_LGB_SEEDS
    lgb_test[horizon] = all_test / N_LGB_SEEDS
    b = compute_brier(time_values, event_values, np.clip(lgb_oof[horizon], 0, 1), horizon)
    print(f"  LGB {horizon}h IPCW B@{horizon}={b:.5f}")

GBSA configs: 5 | seeds: 40 | CV-bag test: True
  Config 1/5 done (n=1200, ss=0.7)
  Config 2/5 done (n=1200, ss=0.85)
  Config 3/5 done (n=1200, ss=0.6)
  Config 4/5 done (n=2000, ss=0.85)
  Config 5/5 done (n=1400, ss=0.85)
LGB seeds: 50
  LGB 24h IPCW B@24=0.03047
  LGB 48h IPCW B@48=0.02353


## Blending and Create Submission

In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# Blend + submission
# - 24h/48h are fused GBSA + LGBM
# - 72h uses sigmoid(dist) heuristic
# - Monotonicity enforced before writing CSV
# ─────────────────────────────────────────────────────────────────────────────
# Optional: tune blend weights on OOF (small grid search)
if AUTO_TUNE and DO_OOF:
    best = (-1, None)
    for w24 in TUNE_GRID["W_GBSA_24"]:
        for w48 in TUNE_GRID["W_GBSA_48"]:
            for pcal in TUNE_GRID["POWER_CAL_24"]:
                for thr in TUNE_GRID["SIGMOID_THR"]:
                    for scale in TUNE_GRID["SIGMOID_SCALE"]:
                        oof_tmp = oof_gbsa_raw.copy()
                        oof_tmp[:, 1] = np.clip(oof_tmp[:, 1] ** pcal, 0, 1)
                        oof_tmp[:, 1] = w24 * oof_tmp[:, 1] + (1 - w24) * lgb_oof[24]
                        oof_tmp[:, 2] = w48 * oof_tmp[:, 2] + (1 - w48) * lgb_oof[48]
                        sig = sigmoid_pred(dist_train, thr, scale)
                        if P72_MODE == "blend":
                            p72 = P72_BLEND_W * oof_gbsa_raw[:, 3] + (1 - P72_BLEND_W) * sig
                        elif P72_MODE == "gbsa":
                            p72 = oof_gbsa_raw[:, 3]
                        elif P72_MODE == "constant1":
                            p72 = np.ones_like(sig)
                        else:
                            p72 = sig
                        if P72_FLOOR > 0:
                            p72 = np.clip(p72, P72_FLOOR, 1.0)
                        oof_tmp[:, 3] = p72
                        oof_tmp = enforce_monotonicity(oof_tmp)
                        score, _, _ = compute_hybrid_score(time_values, event_values, oof_tmp[:, 1], oof_tmp[:, 2], oof_tmp[:, 3])
                        if score > best[0]:
                            best = (score, (w24, w48, pcal, thr, scale))
    if best[1] is not None:
        W_GBSA_24, W_GBSA_48, POWER_CAL_24, SIGMOID_THR, SIGMOID_SCALE = best[1]
        print(f"[TUNE] best_hybrid={best[0]:.5f}  W24={W_GBSA_24}  W48={W_GBSA_48}  p24={POWER_CAL_24}  thr={SIGMOID_THR}  scale={SIGMOID_SCALE}")

# Apply power calibration after tuning (or default)
oof_gbsa = oof_gbsa_raw.copy() if DO_OOF else None
test_gbsa = test_gbsa_raw.copy()
if POWER_CAL_24 and POWER_CAL_24 != 1.0:
    if DO_OOF:
        oof_gbsa[:, 1] = np.clip(oof_gbsa[:, 1] ** POWER_CAL_24, 0, 1)
    test_gbsa[:, 1] = np.clip(test_gbsa[:, 1] ** POWER_CAL_24, 0, 1)

if DO_OOF:
    oof_blend = oof_gbsa.copy()
    oof_blend[:, 1] = W_GBSA_24 * oof_gbsa[:, 1] + (1 - W_GBSA_24) * lgb_oof[24]
    oof_blend[:, 2] = W_GBSA_48 * oof_gbsa[:, 2] + (1 - W_GBSA_48) * lgb_oof[48]
    oof_blend[:, 3] = make_p72(dist_train, oof_gbsa[:, 3])
    oof_final = enforce_monotonicity(oof_blend)
    hybrid, c_idx, wb = compute_hybrid_score(
        time_values, event_values, oof_final[:, 1], oof_final[:, 2], oof_final[:, 3]
    )
    b24 = compute_brier(time_values, event_values, oof_final[:, 1], 24)
    b48 = compute_brier(time_values, event_values, oof_final[:, 2], 48)
    b72 = compute_brier(time_values, event_values, oof_final[:, 3], 72)
    print(f"OOF Hybrid: {hybrid:.5f}  C-Index: {c_idx:.4f}  WBrier: {wb:.5f}")
    print(f"  B24={b24:.5f}  B48={b48:.5f}  B72={b72:.5f}")

test_blend = test_gbsa.copy()
test_blend[:, 1] = W_GBSA_24 * test_gbsa[:, 1] + (1 - W_GBSA_24) * lgb_test[24]
test_blend[:, 2] = W_GBSA_48 * test_gbsa[:, 2] + (1 - W_GBSA_48) * lgb_test[48]
test_blend[:, 3] = make_p72(dist_test, test_gbsa[:, 3])
test_final = enforce_monotonicity(test_blend)

submission = pd.DataFrame(
    {
        "event_id": test_df["event_id"].values,
        "prob_12h": test_final[:, 0],
        "prob_24h": test_final[:, 1],
        "prob_48h": test_final[:, 2],
        "prob_72h": test_final[:, 3],
    }
)
submission = sample_sub[["event_id"]].merge(submission, on="event_id", how="left")
submission.to_csv(OUTPUT_PATH, index=False)

print("Saved:", OUTPUT_PATH)
print(submission.head())

OOF Hybrid: 0.97223  C-Index: 0.9416  WBrier: 0.01466
  B24=0.02695  B48=0.01628  B72=0.00020
Saved: /kaggle/working/submission.csv
   event_id  prob_12h  prob_24h  prob_48h  prob_72h
0  10662602  0.015898  0.027341  0.027341  0.600000
1  13353600  0.679577  0.931685  0.979202  0.998654
2  13942327  0.015566  0.026713  0.026713  0.600000
3  16112781  0.724612  0.951102  0.984389  0.999337
4  17132808  0.019712  0.034518  0.034518  0.600000
